# 01 - Environment and Dataset Setup

This notebook sets up the computational environment, installs dependencies, verifies GPU availability, and introduces the MSMARCO-XI dataset. We will load a manageable sample to inspect its schema and structure before diving into full-scale RAG engineering.

## Check Python Version

We verify the Python version to ensure compatibility with our RAG dependencies.

In [ ]:
import sys
print(f'Python version: {sys.version}')

## Check GPU Availability

Checking if a GPU is available and running `nvidia-smi` to view its specifications. This is essential for determining if we can use hardware acceleration for embedding models later.

In [ ]:
!nvidia-smi

## Install Dependencies

Installing the required Python packages from our `requirements-colab.txt`.

In [ ]:
!pip install -r ../requirements-colab.txt -q
print('Dependencies installed.')

## Setup `sys.path`

We need to import from the `colab/src/` directory where our shared utilities and RAG components are stored. We use a relative path logic or find the repository root.

In [ ]:
import sys
import os

notebook_dir = os.path.dirname(os.path.abspath('__file__' if '__file__' in locals() else os.getcwd()))
colab_root = os.path.abspath(os.path.join(notebook_dir, '..'))
if colab_root not in sys.path:
    sys.path.append(colab_root)

print(f'Added to sys.path: {colab_root}')

## Import Utilities and Set Seed

Importing our custom `utils` module and setting a fixed random seed (42) for reproducibility across all our experiments.

In [ ]:
from src import utils

utils.set_seed(42)
print('Seed set to 42 for reproducibility.')

## Load Experiment Configuration

Loading settings from `experiment_config.yaml` to ensure our notebook runs with the correct project-wide parameters.

In [ ]:
config = utils.load_config(os.path.join(colab_root, 'configs', 'experiment_config.yaml'))
print('Loaded configuration.')

## Authenticate with Hugging Face

We authenticate with Hugging Face using the token stored in our secrets. This might be necessary for downloading certain restricted datasets or models.

In [ ]:
try:
    hf_token = utils.get_secret('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('Authenticated with Hugging Face.')
except Exception as e:
    print('Skipping HF authentication. If a dataset is public, this is fine.', e)

## Import Dataset Utilities

Importing functions to interact with the MSMARCO-XI dataset.

In [ ]:
from src import dataset_utils
from datasets import load_dataset_builder
from huggingface_hub import list_datasets

DATASET_NAME = 'msmarco/msmarco-xi'
print(f'Target dataset: {DATASET_NAME}')

## Verify Dataset and List Configurations

Before loading, let's list available languages/splits in MSMARCO-XI to understand our options.

In [ ]:
configs = dataset_utils.list_available_configs(DATASET_NAME) if hasattr(dataset_utils, 'list_available_configs') else []
print(f'Available configs: {configs}')

## Configuration for Sample Loading

We set our variables here. We want to load Hindi (`hi`), use the `validation` split, and limit to 5000 samples.

In [ ]:
LANGUAGE = 'hi'
SPLIT = 'validation'
N_SAMPLES = 5000

print(f'Configured to load {N_SAMPLES} samples from {LANGUAGE} {SPLIT} split.')

## Explain Why NOT to Download the Full Dataset

Downloading the full dataset immediately can be highly inefficient. Datasets like MSMARCO are often massively large, taking significant time and disk space. Loading a small sample first allows us to inspect the schema, develop our processing pipeline, and ensure everything works before committing compute resources to the entire corpus.

In [ ]:
print('Ready to load sample.')

## Load Validation Sample

Loading the dataset based on our configuration.

In [ ]:
ds = dataset_utils.load_msmarco_xi(lang=LANGUAGE, split=SPLIT, streaming=False)
if N_SAMPLES and len(ds) > N_SAMPLES:
    ds = ds.select(range(N_SAMPLES))
print(f'Loaded {len(ds)} samples.')

## Print Dataset Schema and Features

Let's examine the structure of a dataset record.

In [ ]:
print('Dataset Features:')
print(ds.features)

## Display a Complete Sample Record

Viewing the raw data for the very first sample.

In [ ]:
sample_record = ds[0]
print('Sample Record:')
import pprint
pprint.pprint(sample_record)

## Inspect Nested Passage Fields & Verify `is_selected`

The passages are typically provided as a list of dictionaries. We need to check if the `is_selected` field exists which dictates the ground truth for our retrieval task.

In [ ]:
passages = sample_record.get('passages', [])
print(f'Number of passages in sample: {len(passages)}')
if passages:
    print('First passage keys:', passages[0].keys())
    has_is_selected = 'is_selected' in passages[0]
    print(f'`is_selected` field exists: {has_is_selected}')

## Save Dataset Metadata and Environment Info

We save our findings and environment specifications into the `reports/` folder for tracking.

In [ ]:
import json
reports_dir = utils.get_reports_dir()
os.makedirs(reports_dir, exist_ok=True)

metadata = {
    'dataset': DATASET_NAME,
    'language': LANGUAGE,
    'split': SPLIT,
    'loaded_samples': len(ds),
    'features': list(ds.features.keys())
}

utils.save_json(metadata, os.path.join(reports_dir, 'dataset_metadata.json'))

env_info = utils.get_environment_info()
print('Environment Info:', env_info)